# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dwaynemongaya/flyrank-ml-internship_dwaynemongaya/blob/main/work/notebooks/w07_action_playbook.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

### Purpose

The purpose of this action playbook is to help content managers prioritize which pages should be reviewed first based on observable search performance, content characteristics, and the Decision Tree's estimated probability of the declining-trend proxy.

The ranking is intended to support human decision-making rather than automatically determine which pages must be changed.

### Recommended Actions

| Action | Reason Code | When to Recommend |
|---|---|---|
| Review for Decline | DECLINE_SIGNAL | The Decision Tree assigns a relatively high score to the declining-trend proxy. |
| Review Snippet | LOW_CTR_VISIBLE | A page has meaningful impressions but comparatively low CTR. |
| Review Content Freshness | STALE_VISIBLE | A page has not been updated recently but continues receiving search impressions. |
| Monitor Performance | MONITOR | A page has limited evidence for immediate changes and should be observed further. |

### Archetype-to-Action Mapping

The following archetypes are practical groups based on observable signals. They are not outputs from a separate clustering model.

| Archetype | Observed Pattern | Suggested Action |
|---|---|---|
| High-Visibility Pages | High impressions and good average position | PROTECT / MONITOR |
| Potential Declining Pages | High Decision Tree score | REVIEW FOR DECLINE |
| Low-CTR Visible Pages | High impressions but low CTR | REVIEW SNIPPET |
| Stale Visible Pages | Long time since last update and meaningful impressions | REVIEW FRESHNESS |
| Low-Visibility Pages | Limited search impressions | MONITOR |

### Decay and Refresh Insight

The previous analysis suggests that content age, freshness, and search performance are useful signals to consider when reviewing content opportunities.

However, an observed declining trend does not automatically mean that a page needs a refresh. Other explanations, such as seasonality, search-demand changes, and related-page consolidation, should be considered.

A refresh should therefore be recommended only after a human reviewer checks the page's actual condition and identifies a reasonable improvement opportunity.

## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

### Intended Use

The action playbook is designed to help content managers prioritize pages for manual review.

It combines observable content and search signals with the Decision Tree's estimated probability of the declining-trend proxy to create an organized review queue.

The intended users are content managers, SEO analysts, and other team members responsible for evaluating content performance.

The output can help users:

- Identify pages that may deserve a closer review.
- Prioritize pages based on available evidence.
- Understand why a page received a particular recommendation.
- Decide whether to refresh, improve, protect, or monitor a page.

### Limitations

This project has several limitations.

1. The target is based on the current `trend_direction` field, not a separately measured future outcome.
2. The starter dataset is an anonymized sample and does not represent every website or client.
3. Search performance can be influenced by seasonality, competition, search-demand changes, and other external factors.
4. The model's predictions are not proof that a content update will improve performance.
5. The model may produce false positives and false negatives.

Therefore, the playbook should be treated as a decision-support tool and not as a fully automated content optimization system.

## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

### Human Review Rules

Every recommended action must be reviewed by a human before implementation.

The reviewer should:

1. Check whether the page's declining trend is persistent rather than temporary.
2. Examine impressions, clicks, CTR, and average position.
3. Check whether the page has already been updated recently.
4. Consider seasonality and changes in search demand.
5. Review whether the content still satisfies the intended search intent.
6. Decide whether refreshing, rewriting, expanding, protecting, or monitoring is appropriate.

### What Should NOT Be Automated

The following decisions should remain under human control:

- Automatically rewriting or deleting content.
- Automatically publishing changes to a website.
- Automatically changing titles and meta descriptions.
- Automatically merging or removing pages.
- Declaring that a page will recover after a refresh.
- Making business decisions solely from a model score.

### Why Human Review Matters

A high model score does not guarantee that a page needs improvement. The model only identifies patterns associated with its training target.

Human reviewers provide additional context that the dataset may not contain, such as content quality, business priorities, search intent, and recent editorial changes.

For this reason, the playbook recommends pages for review instead of automatically implementing changes.

## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

### Monitoring Plan

The model and action playbook should be reviewed regularly to check whether their recommendations remain useful.

| What to Monitor | Suggested Frequency | Reason |
|---|---|---|
| Search impressions and clicks | Monthly | Identify changes in page visibility and traffic. |
| CTR and average position | Monthly | Observe changes in search performance. |
| Model Precision@K | Monthly or per evaluation cycle | Check whether high-priority recommendations remain relevant. |
| False positives and false negatives | During each evaluation | Understand which recommendations the model gets wrong. |
| Feature distribution changes | Quarterly | Identify possible changes in the data. |

### Retraining Triggers

The model should be considered for retraining when:

1. New labeled observations become available.
2. Feature distributions change substantially.
3. Precision@K consistently decreases across evaluation periods.
4. The model produces more incorrect recommendations than before.
5. A revised target definition or feature set becomes available.

### Retraining Process

Before using an updated model, it should be trained and evaluated again using an appropriate validation design.

The updated model should only replace the existing version if it demonstrates useful improvements against the baseline using the same evaluation data and metrics.

These monitoring frequencies and triggers are proposed operating guidelines, not measured production thresholds. The project remains a research prototype and has not been deployed in a production environment.

## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [1]:

import os
import json
import numpy as np
import pandas as pd

from sklearn.model_selection import GroupShuffleSplit
from sklearn.tree import DecisionTreeClassifier

# 1. Load the dataset
df = pd.read_csv("/content/content_refresh_anonymized.csv")

# 2. Use the same features from Week 5
features = [
    "content_age_days",
    "days_since_last_update",
    "impressions_90d",
    "avg_position",
    "ctr",
    "word_count"
]

X = (
    df[features]
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0)
)

y = (df["trend_direction"] == "down").astype(int)
groups = df["client_id"]

# 3. Grouped-by-client split
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(X, y, groups=groups)
)

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

# 4. Train the Decision Tree
tree = DecisionTreeClassifier(
    max_depth=4,
    class_weight="balanced",
    random_state=42
)

tree.fit(X_train, y_train)

# 5. Generate model scores
tree_scores = tree.predict_proba(X_test)[:, 1]

# 6. Generate baseline scores
baseline_scores = (
    (X_test["impressions_90d"] >= 500).astype(int) * 50
    +
    (X_test["avg_position"] <= 10).astype(int) * 50
)

# 7. Create the recommendation queue
queue = df.iloc[test_idx][
    ["content_id"]
].copy()

queue["model_score"] = tree_scores
queue["baseline_score"] = baseline_scores.values

# 8. Assign reason codes
queue["reason_code"] = np.select(
    [
        queue["model_score"] >= 0.50,
        (
            (X_test["impressions_90d"].values >= 500)
            &
            (X_test["ctr"].values < 0.5)
        ),
        (
            (X_test["days_since_last_update"].values >= 180)
            &
            (X_test["impressions_90d"].values >= 100)
        )
    ],
    [
        "DECLINE_SIGNAL",
        "LOW_CTR_VISIBLE",
        "STALE_VISIBLE"
    ],
    default="MONITOR"
)

# 9. Assign suggested actions
action_mapping = {
    "DECLINE_SIGNAL": "REVIEW_FOR_DECLINE",
    "LOW_CTR_VISIBLE": "REVIEW_SNIPPET",
    "STALE_VISIBLE": "REVIEW_FRESHNESS",
    "MONITOR": "MONITOR_PERFORMANCE"
}

queue["action"] = queue["reason_code"].map(action_mapping)

# 10. Assign practical archetypes
queue["archetype"] = np.select(
    [
        queue["reason_code"] == "DECLINE_SIGNAL",
        queue["reason_code"] == "LOW_CTR_VISIBLE",
        queue["reason_code"] == "STALE_VISIBLE",
        (
            (X_test["impressions_90d"].values >= 500)
            &
            (X_test["avg_position"].values <= 10)
        )
    ],
    [
        "Potential Declining Pages",
        "Low-CTR Visible Pages",
        "Stale Visible Pages",
        "High-Visibility Pages"
    ],
    default="Low-Visibility or Other Pages"
)

queue["review_required"] = True

# 11. Rank the recommendations
ranked_queue = queue.sort_values(
    ["model_score", "baseline_score"],
    ascending=False
).reset_index(drop=True)

ranked_queue.insert(
    0, "rank",
    range(1, len(ranked_queue) + 1)
)

# 12. Export files
os.makedirs("work/outputs", exist_ok=True)

ranked_queue.to_csv(
    "work/outputs/w07_ranked_action_queue.csv",
    index=False
)

metrics = {
    "dataset_rows": int(len(df)),
    "training_rows": int(len(X_train)),
    "test_rows": int(len(X_test)),
    "training_clients": int(groups.iloc[train_idx].nunique()),
    "test_clients": int(groups.iloc[test_idx].nunique()),
    "client_overlap": int(
        len(
            set(groups.iloc[train_idx])
            &
            set(groups.iloc[test_idx])
        )
    ),
    "decision_tree_depth": int(tree.get_depth()),
    "queue_rows": int(len(ranked_queue)),
    "target": "trend_direction == down",
    "features": features,
    "validation": "Grouped by client",
    "interpretation": "Decision-support prototype"
}

with open(
    "work/outputs/w07_playbook_metrics.json",
    "w"
) as f:
    json.dump(metrics, f, indent=4)

# 13. Display top 20
display(ranked_queue.head(20))

print("Ranked queue exported successfully.")
print("Saved: work/outputs/w07_ranked_action_queue.csv")
print("Saved: work/outputs/w07_playbook_metrics.json")


FileNotFoundError: [Errno 2] No such file or directory: '/content/content_refresh_anonymized.csv'

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.